In [8]:
using LowLevelFEM, LinearAlgebra, IterativeSolvers, Preconditioners, IncompleteLU

In [9]:
openGeometry("tarto.geo")

In [10]:
openPreProcessor()

In [11]:
mat = Material("part", E=2e5, ν=0.3)
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [12]:
A = ∫(U, "hole2", (x,y,z)->1)
Sx = ∫(U, "hole2", (x,y,z)->x)
Sy = ∫(U, "hole2", (x,y,z)->y)
Sz = ∫(U, "hole2", (x,y,z)->z)

xs, ys, zs = Sx/A, Sy/A, Sz/A

(4.005804115633015e-6, -8.595785970749574e-7, 130.0000269311167)

In [13]:
Premote = gmsh.model.occ.addPoint(xs, ys, zs)
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(0, [Premote], -1, "remote")

gmsh.model.mesh.clear()
gmsh.model.mesh.generate(3)

## Anyagállandók megadása, feladat kitűzése

In [14]:
mat = Material("part", E=2e5, ν=0.3)
rem = Material("remote")
U = Field([mat, rem], type=:VectorField, dim=3, fieldName=:u)

ndofs(U)

82173

## Az erő támadáspontja a furat súlypontjában

In [16]:
K = ∫(SymGrad(U)' ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="part");
K[:,:]

82173×82173 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6107383 stored entries:
⎡⡿⣯⡹⠟⠀⠛⠿⠿⠧⢿⣅⢨⡅⣩⡀⣈⢈⣉⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎤
⎢⣷⠎⢿⣷⣶⣶⣶⡆⠀⠀⠀⠈⠁⠉⠉⠉⠈⠉⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣤⠀⢸⣿⠿⣧⡅⠄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⡿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⡇⠸⠿⠁⠍⢻⣶⣆⡆⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣯⣷⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣭⣇⠀⠀⠀⠀⠨⠽⠿⣧⣀⠀⠀⠀⠀⠀⠀⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⎥
⎢⡁⣙⡀⠀⠀⠀⠀⠀⠀⠘⣿⣿⣃⡛⠀⣀⢘⣻⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⡅⣩⡅⠀⠀⠀⠀⠀⠀⠀⣭⠸⠿⣧⣤⣭⢨⣭⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣽⎥
⎢⡀⢨⡇⠀⠀⠀⠀⠀⠀⠀⠀⢠⡄⣿⡿⣯⣼⣭⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⡆⢰⡆⠀⠀⠀⠀⠀⠀⠀⣶⣰⡆⣶⡖⣿⣻⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⡿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣯⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎣⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣟⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎦

In [17]:
f = ∫(U ⋅ [0, -1000, 0], Γ="remote");

In [18]:
supp1 = BoundaryCondition("base", uz=0, field=U)
supp2 = BoundaryCondition("hole1", ux=0, uy=0, field=U)
supp3 = BoundaryCondition("perim", uz=-0.00001, field=U);

In [19]:
mpc_u = MultiPointConstraint(master="remote", slave="hole2", field=U);

In [20]:
u = solveField(Symmetric(K), f, support=[supp1, supp2, supp3], mpc=[mpc_u]);

In [21]:
showDoFResults(f, name="f")
showDoFResults(u, name="u", visible=true, factor=1);

In [22]:
u = nodesToElements(u, onPhysicalGroup="part");

In [23]:
ε = (u ∘ ∇ + ∇ ∘ u) / 2
showStrainResults(ε, name="ε");

In [24]:
E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν/(1-2ν) * trace(ε) * I)

showDoFResults(σ, name="σ");

In [25]:
λ, Q = eigen(σ);

In [26]:
λ1 = λ[1]
λ2 = λ[2]
λ3 = λ[3];

In [27]:
σ_Mohr = λ1 - λ3
showElementResults(σ_Mohr, name="Mohr");

In [28]:
σ_HMH = √(((λ1 - λ2)^2 + (λ2 - λ3)^2 + (λ3 - λ1)^2) / 2)
showElementResults(σ_HMH, name="von Mises");

In [29]:
showElementResults(σ, :z, name="σz")
showElementResults(σ_Mohr - σ_HMH, name="diff")
showDoFResults((σ_Mohr - σ_HMH) / σ_Mohr * 100, name="rel. diff - smooth")

8

In [30]:
b = -σ⋅∇
showElementResults(b, name="b")

9

In [31]:
openPostProcessor()